In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_default_day2.csv")

df.head()
df.shape
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 5350 entries, 0 to 5349
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   loan_id              5350 non-null   int64  
 1   applicant_name       5350 non-null   str    
 2   age                  5350 non-null   int64  
 3   employment_type      5350 non-null   str    
 4   employment_years     5350 non-null   int64  
 5   monthly_income       4595 non-null   float64
 6   loan_amount          5350 non-null   float64
 7   loan_date            5350 non-null   str    
 8   credit_score         5350 non-null   int64  
 9   region               5350 non-null   str    
 10  collection_calls     5350 non-null   int64  
 11  recovery_amount      5350 non-null   float64
 12  account_closed_date  937 non-null    str    
 13  risk_flag            5350 non-null   str    
 14  late_but_repaid      5350 non-null   str    
 15  defaulted_12m        5350 non-null   int64  
dtyp

,loan_id,age,employment_years,monthly_income,loan_amount,credit_score,collection_calls,recovery_amount,defaulted_12m
count,5350.000000,5350.000000,5350.000000,4.595000e+03,5.350000e+03,5350.000000,5350.000000,5350.000000,5350.000000
mean,12569.532150,37.944112,76.188785,3.986177e+04,4.341793e+05,681.059626,1.291963,24892.635514,0.219252
std,1487.996407,9.801472,240.216829,1.482944e+05,2.869913e+05,70.481009,2.711323,75527.705883,0.413779
min,10001.000000,0.000000,0.000000,0.000000e+00,2.400000e+04,436.000000,0.000000,0.000000,0.000000
25%,11280.250000,31.000000,7.000000,2.400000e+04,2.250000e+05,633.000000,0.000000,0.000000,0.000000
50%,12569.500000,38.000000,14.000000,3.520000e+04,3.665000e+05,681.000000,0.000000,0.000000,0.000000
75%,13860.750000,45.000000,22.000000,4.870000e+04,5.827500e+05,727.000000,1.000000,0.000000,0.000000
max,15150.000000,65.000000,999.000000,9.999999e+06,2.481000e+06,900.000000,17.000000,970100.000000,1.000000


In [14]:
df["employment_type"].value_counts()
# df["region"].value_counts()
# df["risk_flag"].value_counts()
# df["late_but_repaid"].value_counts()

employment_type
Salaried         2117
Business         1073
Daily wage        838
Self-employed     836
Self employed     140
Self-Employed     124
SE                117
self employed     105
Name: count, dtype: int64

In [17]:
df.isna().mean() * 100
df.isna().sum()
df.groupby(df["monthly_income"].isna())["defaulted_12m"].mean()

monthly_income
False    0.163221
True     0.560265
Name: defaulted_12m, dtype: float64

In [21]:
data = df.copy()
data["loan_date"] = pd.to_datetime(data["loan_date"])
data["account_closed_date"] = pd.to_datetime(data["account_closed_date"])
data["loan_date"] = data["loan_date"].map(lambda x: x.toordinal())
data["account_closed_date"] = data["account_closed_date"].map(
    lambda x: x.toordinal() if pd.notna(x) else np.nan
)
X = data.drop("defaulted_12m", axis=1)
y = data["defaulted_12m"]
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

In [22]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

model = GradientBoostingClassifier(random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9280149606606983)

In [23]:
X1 = X.drop("collection_calls", axis=1)

scores = cross_val_score(
    model,
    X1,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9269244087360977)

In [24]:
X2 = X.drop("recovery_amount", axis=1)

scores = cross_val_score(
    model,
    X2,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9241916324751885)

In [25]:
X3 = X.drop("account_closed_date", axis=1)

scores = cross_val_score(
    model,
    X3,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9266533446366312)

In [29]:
X4 = X.drop("risk_flag_low", axis=1)

scores = cross_val_score(
    model,
    X4,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9266195312775937)

In [30]:
X5 = X.drop("late_but_repaid_Y", axis=1)

scores = cross_val_score(
    model,
    X5,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9018858604349548)

In [31]:
X6 = X.drop("loan_id", axis=1)

scores = cross_val_score(
    model,
    X6,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9280111772346686)

In [32]:
df.duplicated().sum()

np.int64(200)

In [33]:
df[df.duplicated(keep=False)]

,loan_id,applicant_name,age,employment_type,employment_years,monthly_income,loan_amount,loan_date,credit_score,region,collection_calls,recovery_amount,account_closed_date,risk_flag,late_but_repaid,defaulted_12m
26,12616,Ravi Rathod,43,Salaried,20,51300.0,407000.0,2022-11-12,679,Central Gujarat,0,0.0,NaN,low,N,1
27,10208,Bhavesh Vora,31,Business,3,57000.0,993000.0,2021-10-19,742,Saurashtra,0,0.0,NaN,low,N,0
42,11354,Rekha Vora,49,Salaried,21,33100.0,287000.0,2023-03-01,615,South Gujarat,0,0.0,NaN,high,N,0
44,14822,Bhavesh Patel,35,Salaried,7,26400.0,104000.0,2021-02-10,676,South Gujarat,9,57800.0,2021-12-06,high,N,1
55,13102,Nisha Vora,43,Daily wage,21,18200.0,287000.0,2023-04-23,626,Central Gujarat,0,0.0,NaN,low,N,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5312,10815,Pooja Jadeja,53,Salaried,27,58300.0,358000.0,2021-09-28,696,Saurashtra,1,0.0,NaN,low,N,0
5317,14551,Harsh Trivedi,37,Self employed,11,NaN,615000.0,2022-09-02,626,Central Gujarat,8,226900.0,2023-09-12,high,N,1
5329,14326,Jignesh Jadeja,22,Business,0,60200.0,728000.0,2021-09-08,817,Central Gujarat,0,0.0,NaN,low,N,0
5332,13168,Amit Trivedi,31,Business,10,61800.0,588000.0,2022-03-24,745,South Gujarat,1,0.0,NaN,low,Y,0


In [34]:
temp = df.copy()

temp["name_clean"] = (
    temp["applicant_name"]
    .str.strip()
    .str.lower()
)

near = temp[temp.duplicated(
    subset=["name_clean", "age", "employment_type",
            "employment_years", "loan_amount",
            "credit_score", "region"],
    keep=False
)]

near.shape

(693, 17)

In [35]:
suspicious = df[
    ((df["collection_calls"] > 0) & (df["defaulted_12m"] == 0)) |
    ((df["recovery_amount"] > 0) & (df["defaulted_12m"] == 0)) |
    ((df["risk_flag"] == "high") & (df["defaulted_12m"] == 0))
]

suspicious[[
    "loan_id",
    "collection_calls",
    "recovery_amount",
    "risk_flag",
    "late_but_repaid",
    "defaulted_12m"
]].head(20)

,loan_id,collection_calls,recovery_amount,risk_flag,late_but_repaid,defaulted_12m
1,10825,2,0.0,low,N,0
6,11462,1,0.0,low,N,0
8,13134,1,0.0,low,N,0
10,11736,1,0.0,low,N,0
11,13846,1,0.0,low,N,0
12,10135,2,0.0,low,N,0
19,10771,1,0.0,low,N,0
20,11037,1,0.0,low,N,0
22,13791,1,0.0,low,N,0
23,11117,1,0.0,high,N,0


In [36]:
X5 = X.drop("late_but_repaid_Y", axis=1)

scores = cross_val_score(
    model,
    X5,
    y,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9018858604349548)

In [37]:
temp = df.copy()

temp["name_clean"] = (
    temp["applicant_name"]
    .str.strip()
    .str.lower()
)

temp["emp_clean"] = (
    temp["employment_type"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

temp["income_round"] = temp["monthly_income"].round(-3)

keys = [
    "name_clean",
    "age",
    "emp_clean",
    "employment_years",
    "income_round",
    "loan_amount",
    "loan_date",
    "credit_score",
    "region",
    "collection_calls",
    "recovery_amount",
    "account_closed_date",
    "risk_flag",
    "late_but_repaid",
    "defaulted_12m"
]

candidate = temp[temp.duplicated(subset=keys, keep=False)]

candidate.shape

(693, 19)

In [38]:
exact_mask = df.duplicated(keep=False)

near_only = candidate[~exact_mask]

near_only.shape

C:\Users\DELL\AppData\Local\Temp\ipykernel_20184\2635929724.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  near_only = candidate[~exact_mask]


(293, 19)

In [39]:
df_no_dup = df.drop_duplicates().copy()

data2 = df_no_dup.copy()

data2["loan_date"] = pd.to_datetime(data2["loan_date"])
data2["account_closed_date"] = pd.to_datetime(data2["account_closed_date"])

data2["loan_date"] = data2["loan_date"].map(lambda x: x.toordinal())
data2["account_closed_date"] = data2["account_closed_date"].map(
    lambda x: x.toordinal() if pd.notna(x) else np.nan
)

X_no_dup = data2.drop("defaulted_12m", axis=1)
y_no_dup = data2["defaulted_12m"]

X_no_dup = pd.get_dummies(X_no_dup, drop_first=True)
X_no_dup = X_no_dup.fillna(X_no_dup.median(numeric_only=True))

scores = cross_val_score(
    model,
    X_no_dup,
    y_no_dup,
    cv=5,
    scoring="roc_auc"
)

scores.mean()

np.float64(0.9273184167526376)

In [40]:
suspicious = df[
    (
        ((df["recovery_amount"] > 0) & (df["defaulted_12m"] == 0))
        |
        ((df["collection_calls"] >= 5) & (df["defaulted_12m"] == 0))
        |
        ((df["risk_flag"] == "high") & (df["defaulted_12m"] == 0))
    )
]

rows20 = suspicious[[
    "loan_id",
    "collection_calls",
    "recovery_amount",
    "risk_flag",
    "late_but_repaid",
    "defaulted_12m"
]].head(20)

rows20

,loan_id,collection_calls,recovery_amount,risk_flag,late_but_repaid,defaulted_12m
23,11117,1,0.0,high,N,0
38,14576,7,371500.0,high,N,0
42,11354,0,0.0,high,N,0
45,12513,0,0.0,high,N,0
54,14749,6,37300.0,high,N,0
77,13369,0,0.0,high,N,0
91,13978,0,0.0,high,N,0
93,12277,0,0.0,high,Y,0
106,10466,0,0.0,high,N,0
108,11962,0,0.0,high,N,0


In [41]:
rows20 = suspicious[[
    "loan_id",
    "region",
    "collection_calls",
    "recovery_amount",
    "risk_flag",
    "late_but_repaid",
    "defaulted_12m"
]].head(20)

rows20

,loan_id,region,collection_calls,recovery_amount,risk_flag,late_but_repaid,defaulted_12m
23,11117,Saurashtra,1,0.0,high,N,0
38,14576,South Gujarat,7,371500.0,high,N,0
42,11354,South Gujarat,0,0.0,high,N,0
45,12513,Central Gujarat,0,0.0,high,N,0
54,14749,Kutch,6,37300.0,high,N,0
77,13369,Saurashtra,0,0.0,high,N,0
91,13978,Kutch,0,0.0,high,N,0
93,12277,North Gujarat,0,0.0,high,Y,0
106,10466,Kutch,0,0.0,high,N,0
108,11962,Saurashtra,0,0.0,high,N,0


In [42]:
pd.crosstab(
    df["late_but_repaid"],
    [df["region"], df["defaulted_12m"]]
)

region          Central Gujarat      Kutch      North Gujarat      Saurashtra  \
defaulted_12m                 0    1     0    1             0    1          0   
late_but_repaid                                                                 
N                           777  211   621  141           793  220       1211   
Y                            49    0     7   51            56    0         16   

region               South Gujarat       
defaulted_12m      1             0    1  
late_but_repaid                          
N                312           590  140  
Y                 98            57    0